# 中证800 V49 Factor Interaction OOS 实验

目的：在 V48 因子审计基础上，验证“单因子弱，但组合/交互可能有效”的假设。

这版重点修正 V48 的归因问题：

- 因子方向和组合权重只用历史月份估计，不用当前评估月 label。
- 使用互斥 `DISJOINT_GROUPS` 做主要归因，避免 group overlap。
- 测少量有经济含义的因子组交互，不做全组合暴力搜索。
- 不导出 pkl，不训练实盘模型，不 early stopping，不 sample weight。

输出重点：

1. OOS group score：equal / IC-weight / ICIR-weight
2. OOS pair interaction：additive / gated / residual
3. 每个策略的 top10/top30 月度 alpha、胜率、IR、最大回撤、年度表现
4. 哪些组合值得进入下一步模型增强


In [ ]:
import os
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 240)

DATA_PATH = "train_csi800_factor_v40_data_enhancement.csv"
OUT_DIR = "csi800_ml_v49_factor_interaction_oos_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

TARGET_COL = "alpha_1m"
TRAIN_START = "2019-01-01"
TRAIN_END = "2025-03-31"
MIN_HISTORY_MONTHS = 24
TOP_LIST = [10, 30]
WEIGHT_METHODS = ["equal", "ic_weight", "icir_weight"]

print("DATA_PATH =", DATA_PATH)
print("OUT_DIR =", OUT_DIR)
print("target =", TARGET_COL)


In [ ]:
# Main attribution groups are disjoint by design.
DISJOINT_GROUPS = {
    "style_processed": [
        "book_to_price_ratio", "earnings_yield", "liquidity", "momentum",
    ],
    "value_cashflow": [
        "cash_flow_to_price_ratio", "cash_earnings_to_price_ratio", "earnings_to_price_ratio", "sales_to_price_ratio",
    ],
    "quality_profit": [
        "roe_ttm", "roa_ttm", "gross_profit_ttm", "operating_profit_to_total_profit",
        "net_operating_cash_flow_coverage", "adjusted_profit_to_total_profit",
        "operating_profit_per_share", "net_operate_cash_flow_per_share", "total_operating_revenue_per_share",
    ],
    "growth_balance": [
        "ACCA", "growth", "net_working_capital", "MLEV", "debt_to_equity_ratio",
        "debt_to_tangible_equity_ratio", "super_quick_ratio",
    ],
    "momentum_volume": [
        "Rank1M", "sharpe_ratio_60", "VOL10", "DAVOL10", "VMACD", "VOSC",
        "px_close_to_ma60", "ts_Rank1M_rank_chg_1m",
    ],
    "risk_distribution": [
        "Variance20", "beta", "Skewness20", "Kurtosis20", "px_drawdown_60", "liq_paused_count_20",
    ],
    "temporal_liquidity": [
        "liq_money_ratio_20_60", "ts_cash_flow_to_price_ratio_rank_mean_3m",
    ],
}

PAIR_SPECS = [
    ("momentum_volume", "style_processed"),
    ("momentum_volume", "risk_distribution"),
    ("style_processed", "quality_profit"),
    ("style_processed", "temporal_liquidity"),
    ("style_processed", "value_cashflow"),
    ("quality_profit", "value_cashflow"),
]

ALL_FACTOR_COLS = []
for group_name in DISJOINT_GROUPS:
    for col in DISJOINT_GROUPS[group_name]:
        if col in ALL_FACTOR_COLS:
            raise ValueError("duplicate factor in DISJOINT_GROUPS: " + col)
        ALL_FACTOR_COLS.append(col)

print("groups:", len(DISJOINT_GROUPS), "unique factors:", len(ALL_FACTOR_COLS))
for k in sorted(DISJOINT_GROUPS):
    print(k, len(DISJOINT_GROUPS[k]), DISJOINT_GROUPS[k])


In [ ]:
def safe_rank_ic(a, b):
    s = pd.DataFrame({"a": np.asarray(a, dtype=float), "b": np.asarray(b, dtype=float)})
    s = s.replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) < 3 or s["a"].nunique() < 2 or s["b"].nunique() < 2:
        return np.nan
    return s["a"].rank(pct=True).corr(s["b"].rank(pct=True))


def safe_stats(values):
    s = pd.Series(values).replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) == 0:
        return {"mean": np.nan, "std": np.nan, "ir": np.nan, "hit_rate": np.nan, "months": 0}
    std = s.std()
    return {
        "mean": float(s.mean()),
        "std": float(std) if not pd.isnull(std) else np.nan,
        "ir": float(s.mean() / std) if (not pd.isnull(std) and std > 0) else np.nan,
        "hit_rate": float((s > 0).mean()),
        "months": int(len(s)),
    }


def max_drawdown_from_returns(returns):
    s = pd.Series(returns).replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) == 0:
        return np.nan
    nav = (1.0 + s).cumprod()
    dd = nav / nav.cummax() - 1.0
    return float(dd.min())


def safe_monthly_rank_pct(df, value_col):
    out = pd.Series(index=df.index, dtype=float)
    for _, idx in df.groupby("rebalance_date").groups.items():
        s = pd.to_numeric(df.loc[idx, value_col], errors="coerce").replace([np.inf, -np.inf], np.nan)
        valid = s.dropna()
        if len(valid) == 0:
            continue
        out.loc[valid.index] = valid.rank(method="average") / float(len(valid))
    return out


def cross_section_rank_pct(s):
    s = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)
    out = pd.Series(index=s.index, dtype=float)
    valid = s.dropna()
    if len(valid) == 0:
        return out
    out.loc[valid.index] = valid.rank(method="average") / float(len(valid))
    return out


def calc_monthly_ic_table(hist_df, factors):
    rows = []
    for dt, g in hist_df.groupby("rebalance_date"):
        for factor in factors:
            if factor not in g.columns:
                continue
            rows.append({"rebalance_date": dt, "factor": factor, "rank_ic": safe_rank_ic(g[factor], g[TARGET_COL])})
    return pd.DataFrame(rows)


def summarize_prior_ic(hist_df, factors):
    ic_df = calc_monthly_ic_table(hist_df, factors)
    rows = []
    for factor in factors:
        s = ic_df[ic_df["factor"] == factor]["rank_ic"] if not ic_df.empty else pd.Series(dtype=float)
        st = safe_stats(s)
        rows.append({"factor": factor, "ic_mean": st["mean"], "ic_std": st["std"], "ic_ir": st["ir"], "months": st["months"]})
    return pd.DataFrame(rows)


def get_factor_weights(prior_df, factors, method):
    factors = [f for f in factors if f in prior_df["factor"].values]
    if len(factors) == 0:
        return {}
    m = prior_df.set_index("factor")
    raw = {}
    for f in factors:
        ic_mean = m.loc[f, "ic_mean"]
        ic_std = m.loc[f, "ic_std"]
        if pd.isnull(ic_mean):
            raw[f] = 0.0
        elif method == "equal":
            raw[f] = 1.0 if ic_mean >= 0 else -1.0
        elif method == "ic_weight":
            raw[f] = float(ic_mean)
        elif method == "icir_weight":
            if pd.isnull(ic_std) or ic_std <= 0:
                raw[f] = 0.0
            else:
                raw[f] = float(ic_mean / ic_std)
        else:
            raise ValueError("unknown weight method: " + str(method))
    denom = sum(abs(v) for v in raw.values())
    if denom <= 0:
        return {f: 0.0 for f in factors}
    return {f: raw[f] / denom for f in factors}


def score_group_month(month_df, factors, weights):
    parts = []
    for f in factors:
        if f not in month_df.columns or f not in weights:
            continue
        w = weights.get(f, 0.0)
        if w == 0:
            continue
        r = cross_section_rank_pct(month_df[f])
        parts.append((r - 0.5) * 2.0 * w)
    if len(parts) == 0:
        return pd.Series(index=month_df.index, dtype=float)
    return pd.concat(parts, axis=1).sum(axis=1)


def linear_residual(y, x):
    d = pd.DataFrame({"y": y, "x": x}).replace([np.inf, -np.inf], np.nan).dropna()
    out = pd.Series(index=y.index, dtype=float)
    if len(d) < 5 or d["x"].nunique() < 2:
        out.loc[d.index] = d["y"]
        return out
    x0 = d["x"].values.astype(float)
    y0 = d["y"].values.astype(float)
    x_mean = x0.mean()
    y_mean = y0.mean()
    denom = ((x0 - x_mean) ** 2).sum()
    if denom <= 0:
        out.loc[d.index] = d["y"]
        return out
    beta = ((x0 - x_mean) * (y0 - y_mean)).sum() / denom
    alpha = y_mean - beta * x_mean
    out.loc[d.index] = y0 - (alpha + beta * x0)
    return out


def eval_topn(month_df, score_col, n):
    d = month_df[["stock", "rebalance_date", TARGET_COL, score_col]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(d) == 0:
        return None
    top = d.sort_values(score_col, ascending=False).head(min(n, len(d)))
    return {"mean_alpha": float(top[TARGET_COL].mean()), "count": int(len(top)), "targets": ",".join(list(top["stock"]))}


In [ ]:
def load_df(path):
    if not os.path.exists(path):
        raise IOError("DATA_PATH not found: " + path)
    df = pd.read_csv(path)
    if "code" in df.columns and "stock" not in df.columns:
        df = df.rename(columns={"code": "stock"})
    for col in ["rebalance_date", "feature_date", "next_date"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col]).dt.normalize()
    if TARGET_COL not in df.columns:
        raise ValueError("missing target: " + TARGET_COL)
    df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
    df = df.dropna(subset=["stock", "rebalance_date", TARGET_COL]).copy()
    return df


df_all = load_df(DATA_PATH)
work_df = df_all[(df_all["rebalance_date"] >= pd.Timestamp(TRAIN_START)) & (df_all["rebalance_date"] <= pd.Timestamp(TRAIN_END))].copy()
available_factors = [c for c in ALL_FACTOR_COLS if c in work_df.columns]
missing_factors = [c for c in ALL_FACTOR_COLS if c not in work_df.columns]
months = sorted(pd.to_datetime(work_df["rebalance_date"].dropna().unique()))
eval_months = months[MIN_HISTORY_MONTHS:]

print("all:", df_all.shape, df_all["rebalance_date"].min(), df_all["rebalance_date"].max())
print("work:", work_df.shape, work_df["rebalance_date"].min(), work_df["rebalance_date"].max(), "months", len(months))
print("eval months:", len(eval_months), eval_months[0] if eval_months else None, eval_months[-1] if eval_months else None)
print("available factors:", len(available_factors))
print(available_factors)
print("missing factors:", missing_factors)


In [ ]:
def build_oos_month_scores(work_df, eval_months):
    strategy_rows = []
    factor_weight_rows = []
    month_score_parts = []

    for mi, month in enumerate(eval_months):
        hist_df = work_df[work_df["rebalance_date"] < month].copy()
        month_df = work_df[work_df["rebalance_date"] == month].copy()
        if hist_df["rebalance_date"].nunique() < MIN_HISTORY_MONTHS or month_df.empty:
            continue
        prior_ic = summarize_prior_ic(hist_df, available_factors)
        score_df = month_df[["stock", "rebalance_date", TARGET_COL]].copy()

        for group_name in sorted(DISJOINT_GROUPS):
            factors = [f for f in DISJOINT_GROUPS[group_name] if f in available_factors]
            for method in WEIGHT_METHODS:
                weights = get_factor_weights(prior_ic, factors, method)
                score_col = "group__{}__{}".format(group_name, method)
                score_df[score_col] = score_group_month(month_df, factors, weights)
                strategy_rows.append({"strategy_name": score_col, "strategy_type": "group", "rebalance_date": month})
                for f, w in weights.items():
                    factor_weight_rows.append({"rebalance_date": month, "strategy_name": score_col, "factor": f, "weight": w})

        for a, b in PAIR_SPECS:
            for method in WEIGHT_METHODS:
                col_a = "group__{}__{}".format(a, method)
                col_b = "group__{}__{}".format(b, method)
                if col_a not in score_df.columns or col_b not in score_df.columns:
                    continue
                pair = a + "__" + b + "__" + method

                score_df["pair_add__" + pair] = (score_df[col_a] + score_df[col_b]) / 2.0
                strategy_rows.append({"strategy_name": "pair_add__" + pair, "strategy_type": "pair_add", "rebalance_date": month})

                # Gate: top 40% by A, rank by B inside the gate.
                gate_a = cross_section_rank_pct(score_df[col_a]) >= 0.60
                score_df["pair_gate_{}_then_{}__{}".format(a, b, method)] = score_df[col_b].where(gate_a)
                strategy_rows.append({"strategy_name": "pair_gate_{}_then_{}__{}".format(a, b, method), "strategy_type": "pair_gate", "rebalance_date": month})

                gate_b = cross_section_rank_pct(score_df[col_b]) >= 0.60
                score_df["pair_gate_{}_then_{}__{}".format(b, a, method)] = score_df[col_a].where(gate_b)
                strategy_rows.append({"strategy_name": "pair_gate_{}_then_{}__{}".format(b, a, method), "strategy_type": "pair_gate", "rebalance_date": month})

                resid_b = linear_residual(score_df[col_b], score_df[col_a])
                score_df["pair_resid_{}_plus_{}__{}".format(a, b, method)] = score_df[col_a] + resid_b
                strategy_rows.append({"strategy_name": "pair_resid_{}_plus_{}__{}".format(a, b, method), "strategy_type": "pair_resid", "rebalance_date": month})

        month_score_parts.append(score_df)
        if (mi + 1) % 12 == 0:
            print("processed eval months:", mi + 1)

    score_all = pd.concat(month_score_parts, ignore_index=True) if month_score_parts else pd.DataFrame()
    return score_all, pd.DataFrame(strategy_rows), pd.DataFrame(factor_weight_rows)


oos_score_df, strategy_month_df, factor_weight_df = build_oos_month_scores(work_df, eval_months)
print("oos_score_df:", oos_score_df.shape)
print("strategies:", len([c for c in oos_score_df.columns if c not in ["stock", "rebalance_date", TARGET_COL]]))
print("factor_weight_df:", factor_weight_df.shape)


In [ ]:
def summarize_strategy_monthly(monthly_df):
    rows = []
    for (strategy_name, topn), g in monthly_df.groupby(["strategy_name", "topn"]):
        st = safe_stats(g["mean_alpha"])
        rows.append({
            "strategy_name": strategy_name,
            "strategy_type": g["strategy_type"].iloc[0],
            "topn": int(topn),
            "months": st["months"],
            "mean_monthly_alpha": st["mean"],
            "monthly_alpha_ir": st["ir"],
            "win_rate": st["hit_rate"],
            "max_drawdown": max_drawdown_from_returns(g.sort_values("rebalance_date")["mean_alpha"]),
        })
    return pd.DataFrame(rows)


monthly_rows = []
score_cols = [c for c in oos_score_df.columns if c not in ["stock", "rebalance_date", TARGET_COL]]
for score_col in score_cols:
    stype = "group"
    if score_col.startswith("pair_add__"):
        stype = "pair_add"
    elif score_col.startswith("pair_gate_"):
        stype = "pair_gate"
    elif score_col.startswith("pair_resid_"):
        stype = "pair_resid"
    for dt, g in oos_score_df.groupby("rebalance_date"):
        for n in TOP_LIST:
            r = eval_topn(g, score_col, n)
            if r is None:
                continue
            monthly_rows.append({
                "rebalance_date": dt,
                "strategy_name": score_col,
                "strategy_type": stype,
                "topn": int(n),
                "mean_alpha": r["mean_alpha"],
                "count": r["count"],
                "targets": r["targets"],
            })

strategy_monthly_df = pd.DataFrame(monthly_rows)
strategy_summary_df = summarize_strategy_monthly(strategy_monthly_df)
strategy_summary_df = strategy_summary_df.sort_values(["topn", "mean_monthly_alpha"], ascending=[True, False])

strategy_monthly_df.to_csv(os.path.join(OUT_DIR, "v49_strategy_monthly_alpha.csv"), index=False)
strategy_summary_df.to_csv(os.path.join(OUT_DIR, "v49_strategy_summary.csv"), index=False)
factor_weight_df.to_csv(os.path.join(OUT_DIR, "v49_factor_weights_oos.csv"), index=False)
oos_score_df.to_csv(os.path.join(OUT_DIR, "v49_oos_scores.csv"), index=False)

print("top10 summary top 30:")
print(strategy_summary_df[strategy_summary_df["topn"] == 10].head(30).to_string(index=False))
print("top30 summary top 20:")
print(strategy_summary_df[strategy_summary_df["topn"] == 30].head(20).to_string(index=False))


In [ ]:
# Yearly summary for top candidates.
top_candidates = list(strategy_summary_df[strategy_summary_df["topn"] == 10].head(20)["strategy_name"])
year_rows = []
if not strategy_monthly_df.empty:
    tmp = strategy_monthly_df[strategy_monthly_df["strategy_name"].isin(top_candidates)].copy()
    tmp["year"] = pd.to_datetime(tmp["rebalance_date"]).dt.year
    for (strategy_name, topn, year), g in tmp.groupby(["strategy_name", "topn", "year"]):
        st = safe_stats(g["mean_alpha"])
        year_rows.append({
            "strategy_name": strategy_name,
            "strategy_type": g["strategy_type"].iloc[0],
            "topn": int(topn),
            "year": int(year),
            "months": st["months"],
            "mean_alpha": st["mean"],
            "ir": st["ir"],
            "win_rate": st["hit_rate"],
        })

yearly_summary_df = pd.DataFrame(year_rows).sort_values(["strategy_name", "topn", "year"])
yearly_summary_df.to_csv(os.path.join(OUT_DIR, "v49_yearly_summary_top_candidates.csv"), index=False)
print(yearly_summary_df.head(80).to_string(index=False))


In [ ]:
# Compare group-only versus pair strategies to spot true interaction value.
compare_rows = []
summary10 = strategy_summary_df[strategy_summary_df["topn"] == 10].copy()
summary10_map = summary10.set_index("strategy_name")
for a, b in PAIR_SPECS:
    for method in WEIGHT_METHODS:
        base_a = "group__{}__{}".format(a, method)
        base_b = "group__{}__{}".format(b, method)
        base_best = np.nan
        if base_a in summary10_map.index and base_b in summary10_map.index:
            base_best = max(summary10_map.loc[base_a, "mean_monthly_alpha"], summary10_map.loc[base_b, "mean_monthly_alpha"])
        candidates = [
            "pair_add__{}__{}__{}".format(a, b, method),
            "pair_gate_{}_then_{}__{}".format(a, b, method),
            "pair_gate_{}_then_{}__{}".format(b, a, method),
            "pair_resid_{}_plus_{}__{}".format(a, b, method),
        ]
        for sname in candidates:
            if sname not in summary10_map.index:
                continue
            row = summary10_map.loc[sname]
            compare_rows.append({
                "pair": a + "+" + b,
                "method": method,
                "strategy_name": sname,
                "strategy_type": row["strategy_type"],
                "mean_monthly_alpha": row["mean_monthly_alpha"],
                "monthly_alpha_ir": row["monthly_alpha_ir"],
                "win_rate": row["win_rate"],
                "max_drawdown": row["max_drawdown"],
                "best_single_group_alpha": base_best,
                "alpha_lift_vs_best_group": row["mean_monthly_alpha"] - base_best if not pd.isnull(base_best) else np.nan,
            })

interaction_lift_df = pd.DataFrame(compare_rows).sort_values("alpha_lift_vs_best_group", ascending=False)
interaction_lift_df.to_csv(os.path.join(OUT_DIR, "v49_interaction_lift_vs_group.csv"), index=False)
print(interaction_lift_df.head(40).to_string(index=False))


In [ ]:
summary_xlsx = os.path.join(OUT_DIR, "v49_factor_interaction_oos_summary.xlsx")
try:
    with pd.ExcelWriter(summary_xlsx) as writer:
        strategy_summary_df.to_excel(writer, "strategy_summary", index=False)
        interaction_lift_df.to_excel(writer, "interaction_lift", index=False)
        yearly_summary_df.to_excel(writer, "yearly_top_candidates", index=False)
        factor_weight_df.to_excel(writer, "factor_weights", index=False)
    print("saved xlsx:", summary_xlsx)
except Exception as err:
    print("xlsx export skipped:", err)

print("saved files:")
for fn in sorted(os.listdir(OUT_DIR)):
    print("  ", fn)


## 结论填写区

重点回填：

- OOS group score 是否复现 V48 的组强弱？
- 哪些 pair interaction 相比最强单组有正 lift？
- additive / gate / residual 哪种更稳定？
- 是否只在某一年有效？
- 哪些组合值得进入下一步模型特征增强？

失败标准：

- OOS top10 alpha 没超过对应最强单组：不认为有交互价值。
- 只在单一年份有效：标记 regime，不作为主线。
- top10 高但 top30 很差：可能过于尾部噪声，谨慎。
